# S02-Submission 01_Graphs

In [11]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

In [12]:
# Import necessary modules from topologicpy

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color
from topologicpy.Plotly import Plotly


renderer = "vscode"

In [13]:
# Update topologicpy to the latest version

import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "topologicpy"])
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())



This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.22) is EQUAL TO the latest version available on PyPI.


## 0. Import the OBJ file

In [14]:
# Import the .obj geometry as topologies and create a house with appertures

objects = Topology.ByOBJPath(r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-01_RamonGarcia\Assets\Assignment-01_Grasshopper_Walls_Auto.obj", selfMerge=True)
print("objects:", objects)
print(type(objects))
print(f"Total objects: {len(objects)}")

window_c = Topology.ByOBJPath(r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-01_RamonGarcia\Assets\Assignment-01_Grasshopper_Windows_Auto.obj", selfMerge=True)
door_c = Topology.ByOBJPath(r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-01_RamonGarcia\Assets\Assignment-01_Grasshopper_Doors_Auto.obj", selfMerge=True)
print(f"Total windows: {len(window_c)}")
print(f"Total doors: {len(door_c)}")

aperture_f = (window_c) + (door_c)
print("aperture_f:", aperture_f)
print(f"Total aperture faces: {len(aperture_f)}")

objects: [<topologic_core.Cluster object at 0x0000029184A5C030>, <topologic_core.Cell object at 0x0000029184AF32F0>, <topologic_core.Cell object at 0x0000029184AF3DB0>, <topologic_core.Cell object at 0x0000029184AF3BF0>, <topologic_core.Cell object at 0x0000029184AF3CF0>, <topologic_core.Cell object at 0x0000029184AF03F0>, <topologic_core.Cell object at 0x00000291FB6BA2F0>, <topologic_core.Cell object at 0x00000291BA5337B0>, <topologic_core.Cell object at 0x0000029184ABD930>, <topologic_core.Cell object at 0x0000029184ABDDB0>, <topologic_core.Cell object at 0x0000029184ABE1F0>, <topologic_core.Cell object at 0x0000029184ABDF70>, <topologic_core.Cell object at 0x00000291FB64C3B0>, <topologic_core.Cell object at 0x00000291BA55B670>, <topologic_core.Cell object at 0x0000029184AEEB70>, <topologic_core.Cell object at 0x0000029184AEC9F0>, <topologic_core.Cell object at 0x0000029184AEC770>, <topologic_core.Cell object at 0x0000029184AEC330>, <topologic_core.Cell object at 0x0000029184AED3F0>,

In [15]:
# Visualize the house with appertures

Topology.Show(objects, aperture_f,
              faceColor=[238,238,238],
              faceOpacity=0.3,
              edgeColor="gray",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="white",
              width=800,
              height=600,
              renderer = renderer)

print(renderer)

vscode


In [16]:
# Create a cell for each room and assign a color based on the name of the room

cells = []
selectors = []

for object in objects:
    d = Topology.Dictionary(object)
    faces = Topology.Faces(object)

    if len(faces) > 1:
        c = Cell.ByFaces(faces, tolerance=0.001)
        c = Topology.RemoveCollinearEdges(c)
        s = Topology.InternalVertex(c)
        name = Dictionary.ValueAtKey(d, "name")
        if "Living" in name or "Room" in name or "Master" in name:
            color = "blue"
        elif "Corridor" in name:
            color = "red"
        elif "Stairs" in name:
            color = "green"
        elif "Service" in name:
            color = "yellow"
        else: 
            color = "gray"
            
        d = Dictionary.SetValuesAtKeys(d, ["color", "vertex_size"], [color, 20])
        s = Topology.SetDictionary(s, d)
        selectors.append(s)
        cells.append(c)

# Create a cell complex for the building and transfer dictionaries

building = CellComplex.ByCells(cells)
building = Topology.TransferDictionariesBySelectors(building, selectors, tranCells=True)
building_cells = Topology.Cells(building)

In [17]:
import plotly.graph_objects as go

# show the house with the assigned colors to each room (with transparency)
color_groups = {"blue": [], "red": [], "green": [], "yellow": [], "gray": []}

for c in building_cells:
    d = Topology.Dictionary(c)
    color = Dictionary.ValueAtKey(d, "color") or "gray"
    color_groups[color].append(c)

all_data = []
for color, cells in color_groups.items():
    if not cells:
        continue
    for cell in cells:
        data = Plotly.DataByTopology(
            topology=cell,
            showVertices=False,
            showEdges=True,
            edgeColor="black",
            edgeWidth=1,
            faceColor=color,
            faceOpacity=0.3,
        )
        all_data.extend(data)

fig = go.Figure(data=all_data)
fig.update_layout(
    width=800, height=600,
    paper_bgcolor="white",
    plot_bgcolor="white",
    scene=dict(
        bgcolor="white",
        xaxis=dict(showticklabels=False, title=""),
        yaxis=dict(showticklabels=False, title=""),
        zaxis=dict(showticklabels=False, title=""),
    )
)
fig.show(renderer=renderer)


In [18]:
# create appertures in the building from imported geometry .obj

building = Topology.AddApertures(building, aperture_f, subTopologyType="face")
print("building:", building)

building: <topologic_core.CellComplex object at 0x0000029184AE4170>


## 1. Derive graph by cell's colours

In [19]:
# Create a graph from the building topology & visualize the graph with the building and appertures

g1 = Graph.ByTopology(building, direct=False, viaSharedApertures=True, toExteriorApertures=True)

vertices = Graph.Vertices(g1)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "black"])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(g1)
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "gray"])
    e = Topology.SetDictionary(e, d)


Topology.Show(building_cells, aperture_f, g1, 
              faceColorKey="color",
              vertexSizeKey="vertex_size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              edgeWidth=2,
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer = renderer) 

## 2 Derive the access graph with appertures

In [20]:
# Create a graph from the building topology & visualize the graph with the building and appertures

g1 = Graph.ByTopology(building, direct=False, viaSharedApertures=True, toExteriorApertures=True)

vertices = Graph.Vertices(g1)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "black"])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(g1)
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "gray"])
    e = Topology.SetDictionary(e, d)

# ── Visualización con transparencia ──────────────────────────────────────────
all_data = []

# 1. Rooms — agrupadas por color para que faceOpacity funcione
color_groups = {"blue": [], "red": [], "green": [], "yellow": [], "gray": []}
for c in building_cells:
    d = Topology.Dictionary(c)
    color = Dictionary.ValueAtKey(d, "color") or "gray"
    color_groups[color].append(c)

for color, cells in color_groups.items():
    if not cells:
        continue
    for cell in cells:
        all_data.extend(Plotly.DataByTopology(
            topology=cell,
            showVertices=False,
            showEdges=True,
            edgeColor="gray",
            faceColor=color,
            faceOpacity=0.06,
        ))

# 2. Apertures — agrupadas en un solo Cluster para no crear 77 traces
ap_cluster = Cluster.ByTopologies(aperture_f)
all_data.extend(Plotly.DataByTopology(
    topology=ap_cluster,
    showVertices=False,
    showEdges=True,
    
    edgeColor="gray",
    edgeWidth=1,
    faceColor=[238, 238, 238],
    faceOpacity=0.3,
))

# 3. Graph vertices — todos negros, misma agrupación
v_cluster = Cluster.ByTopologies(Graph.Vertices(g1))
all_data.extend(Plotly.DataByTopology(
    topology=v_cluster,
    showVertices=True,
    vertexColor="black",
    vertexSize=6,
    showEdges=False,
    showFaces=False,
))

# 4. Graph edges — todos grises
e_cluster = Cluster.ByTopologies(Graph.Edges(g1))
all_data.extend(Plotly.DataByTopology(
    topology=e_cluster,
    showVertices=False,
    showEdges=True,
    edgeColor="gray",
    edgeWidth=2,
    showFaces=False,
))

fig = go.Figure(data=all_data)
fig.update_layout(
    width=500, height=500,
    paper_bgcolor="white",
    scene=dict(
        bgcolor="white",
        xaxis=dict(showticklabels=False, title="", showgrid=False, showbackground=False, zeroline=False),
        yaxis=dict(showticklabels=False, title="", showgrid=False, showbackground=False, zeroline=False),
        zaxis=dict(showticklabels=False, title="", showgrid=False, showbackground=False, zeroline=False),
    )
)

fig.show(renderer=renderer)


## 3 Derive Dual Graph

In [21]:
g2 = Graph.ByTopology(building, direct=True)

In [22]:
vertices = Graph.Vertices(g2)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(g2)
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "black"])
    e = Topology.SetDictionary(e, d)

In [23]:
Topology.Show(building, aperture_f, g2,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

## 4. Derive Primal Graph

In [24]:
# Assamble the graph with vertices and edges from the topology
cluster = Cluster.ByTopologies([house] + aperture_f)
vertices = Topology.Vertices(cluster)
vertices = [Topology.Copy(v) for v in vertices]
edges = Topology.Edges(cluster)
edges = [Topology.Copy(e) for e in edges]
g1 = Graph.ByVerticesEdges(vertices, edges)

NameError: name 'house' is not defined

In [ ]:
# Assign visual attributes to the vertices and edges of the graph
for v in Graph.Vertices(g1):
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size", "color"], [18, "red"]))
for e in Graph.Edges(g1):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [4, "black"]))

In [ ]:
# Show the geometry and the graph
Topology.Show(building, aperture_f, g1,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)